<a href="https://colab.research.google.com/github/cfiscko/rllib-colab/blob/main/Copy_of_RLlib_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Dependencies

If you are running on Google Colab, you need to install the necessary dependencies before beginning the exercise.

In [ ]:
print('NOTE: Intentionally crashing session to use the newly installed library.\n')

!pip uninstall -y pyarrow
!pip install ray
!pip install bs4

!git clone https://github.com/ray-project/tutorial || true
from tutorial.rllib_exercises import test_exercises

print("Successfully installed all the dependencies!")

# A hack to force the runtime to restart, needed to include the above dependencies.
import os
os._exit(0)

NOTE: Intentionally crashing session to use the newly installed library.

Found existing installation: pyarrow 18.1.0
Uninstalling pyarrow-18.1.0:
  Successfully uninstalled pyarrow-18.1.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 MB 10.7 MB/s eta 0:00:00
Cloning into 'tutorial'...
remote: Enumerating objects: 963, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 963 (delta 20), reused 35 (delta 17), pack-reused 921 (from 1)
Receiving objects: 100% (963/963), 31.16 MiB | 40.91 MiB/s, done.
Resolving deltas: 100% (533/533), done.


# RL Exercise - Markov Decision Processes

**GOAL:** The goal of the exercise is to introduce the Markov Decision Process abstraction and to show how to use Markov Decision Processes in Python.

**The key abstraction in reinforcement learning is the Markov decision process (MDP).** An MDP models sequential interactions with an external environment. It consists of the following:
- a **state space**
- a set of **actions**
- a **transition function** which describes the probability of being in a state $s'$ at time $t+1$ given that the MDP was in state $s$ at time $t$ and action $a$ was taken
- a **reward function**, which determines the reward received at time $t$
- a **discount factor** $\gamma$

More details are available [here](https://en.wikipedia.org/wiki/Markov_decision_process).

**NOTE:** Reinforcement learning algorithms are often applied to problems that don't strictly fit into the MDP framework. In particular, situations in which the state of the environment is not fully observed lead to violations of the MDP assumption. Nevertheless, RL algorithms can be applied anyway.

## Policies

A **policy** is a function that takes in a **state** and returns an **action**. A policy may be stochastic (i.e., it may sample from a probability distribution) or it can be deterministic.

The **goal of reinforcement learning** is to learn a **policy** for maximizing the cumulative reward in an MDP. That is, we wish to find a policy $\pi$ which solves the following optimization problem

\begin{equation}
\arg\max_{\pi} \sum_{t=1}^T \gamma^t R_t(\pi),
\end{equation}

where $T$ is the number of steps taken in the MDP (this is a random variable and may depend on $\pi$) and $R_t$ is the reward received at time $t$ (also a random variable which depends on $\pi$).

A number of algorithms are available for solving reinforcement learning problems. Several of the most widely known are [value iteration](https://en.wikipedia.org/wiki/Markov_decision_process#Value_iteration), [policy iteration](https://en.wikipedia.org/wiki/Markov_decision_process#Policy_iteration), and [Q learning](https://en.wikipedia.org/wiki/Q-learning).

## RL in Python

The `gym` Python module provides MDP interfaces to a variety of simulators. For example, the CartPole environment interfaces with a simple simulator which simulates the physics of balancing a pole on a cart. The CartPole problem is described at https://gym.openai.com/envs/CartPole-v0. This example fits into the MDP framework as follows.
- The **state** consists of the position and velocity of the cart as well as the angle and angular velocity of the pole that is balancing on the cart.
- The **actions** are to decrease or increase the cart's velocity by one unit.
- The **transition function** is deterministic and is determined by simulating physical laws.
- The **reward function** is a constant 1 as long as the pole is upright, and 0 once the pole has fallen over. Therefore, maximizing the reward means balancing the pole for as long as possible.
- The **discount factor** in this case can be taken to be 1.

More information about the `gym` Python module is available at https://gym.openai.com/.

In [3]:
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import gymnasium as gym
import numpy as np

The code below illustrates how to create and manipulate MDPs in Python. An MDP can be created by calling `gym.make`. Gym environments are identified by names like `CartPole-v0`. A **catalog of built-in environments** can be found at https://gym.openai.com/envs.

In [4]:
env = gym.make('CartPole-v1')
print('Created env:', env)

Created env: <TimeLimit<OrderEnforcing<PassiveEnvChecker<CartPoleEnv<CartPole-v1>>>>>


Reset the state of the MDP by calling `env.reset()`. This call returns the initial state of the MDP.

In [5]:
state = env.reset()
print('The starting state is:', state)

The starting state is: (array([-0.04274553,  0.01102576,  0.04156822, -0.00808006], dtype=float32), {})


The `env.step` method takes an action (in the case of the CartPole environment, the appropriate actions are 0 or 1, for moving left or right). It returns a tuple of four things:
1. the new state of the environment
2. a reward
3. a boolean indicating whether the simulation has finished
4. a dictionary of miscellaneous extra information

In [7]:
# Simulate taking an action in the environment. Appropriate actions for
# the CartPole environment are 0 and 1 (for moving left and right).
action = 0
state, reward, term, trunc, info = env.step(action)
print(state, reward, term, trunc, info)

[-0.04621836 -0.38035393  0.04735508  0.6028719 ] 1.0 False False {}


A **rollout** is a simulation of a policy in an environment. It alternates between choosing actions based (using some policy) and taking those actions in the environment.

The code below performs a rollout in a given environment. It takes **random actions** until the simulation has finished and returns the cumulative reward.

In [9]:
def random_rollout(env):
    state = env.reset()

    term = trunc = False
    cumulative_reward = 0

    # Keep looping as long as the simulation has not finished.
    while not (term or trunc):
        # Choose a random action (either 0 or 1).
        action = np.random.choice([0, 1])

        # Take the action in the environment.
        state, reward, term, trunc, _ = env.step(action)

        # Update the cumulative reward.
        cumulative_reward += reward

    # Return the cumulative reward.
    return cumulative_reward

reward = random_rollout(env)
print(reward)
reward = random_rollout(env)
print(reward)

13.0
19.0


**EXERCISE:** Finish implementing the `rollout_policy` function below, which should take an environment *and* a policy. The *policy* is a function that takes in a *state* and returns an *action*. The main difference is that instead of choosing a **random action**, the action should be chosen **with the policy** (as a function of the state).

In [14]:
def rollout_policy(env, policy):
    state = env.reset()

    term = trunc = False
    cumulative_reward = 0

    # EXERCISE: Fill out this function by copying the 'random_rollout' function
    # and then modifying it to choose the action using the policy.
    state, _ = env.reset()

    # Keep looping as long as the simulation has not finished.
    while not (term or trunc):
        # Query policy
        action = policy(state)

        # Take the action in the environment.
        state, reward, term, trunc, _ = env.step(action)

        # Update the cumulative reward.
        cumulative_reward += reward

    # Return the cumulative reward.
    return cumulative_reward


    # Return the cumulative reward.
    return cumulative_reward

def sample_policy1(state):
    return 0 if state[0] < 0 else 1

def sample_policy2(state):
    return 1 if state[0] < 0 else 0

reward1 = np.mean([rollout_policy(env, sample_policy1) for _ in range(100)])
reward2 = np.mean([rollout_policy(env, sample_policy2) for _ in range(100)])

print('The first sample policy got an average reward of {}.'.format(reward1))
print('The second sample policy got an average reward of {}.'.format(reward2))

assert 5 < reward1 < 15, ('Make sure that rollout_policy computes the action '
                          'by applying the policy to the state.')
assert 25 < reward2 < 35, ('Make sure that rollout_policy computes the action '
                           'by applying the policy to the state.')

The first sample policy got an average reward of 9.37.
The second sample policy got an average reward of 28.35.


# RLlib Exercise 1 - Proximal Policy Optimization

**GOAL:** The goal of this exercise is to demonstrate how to use the proximal policy optimization (PPO) algorithm.

To understand how to use **RLlib**, see the documentation at http://rllib.io.

PPO is described in detail in https://arxiv.org/abs/1707.06347. It is a variant of Trust Region Policy Optimization (TRPO) described in https://arxiv.org/abs/1502.05477

PPO works in two phases. In one phase, a large number of rollouts are performed (in parallel). The rollouts are then aggregated on the driver and a surrogate optimization objective is defined based on those rollouts. We then use SGD to find the policy that maximizes that objective with a penalty term for diverging too much from the current policy.

![ppo](https://raw.githubusercontent.com/ucbrise/risecamp/risecamp2018/ray/tutorial/rllib_exercises/ppo.png)

**NOTE:** The SGD optimization step is best performed in a data-parallel manner over multiple GPUs. This is exposed through the `num_gpus` field of the `config` dictionary (for this to work, you must be using a machine that has GPUs).

In [1]:
# Be sure to install the latest version of RLlib.
! pip install -U ray[rllib]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 135.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 MB 36.0 MB/s eta 0:00:00
  Attempting uninstall: gymnasium
    Found existing installation: gymnasium 1.2.3
    Uninstalling gymnasium-1.2.3:
      Successfully uninstalled gymnasium-1.2.3


In [2]:
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import gymnasium as gym
import ray
from ray.rllib.algorithms.ppo import PPO, PPOConfig
from ray.tune.logger import pretty_print

In [3]:
# Start up Ray. This must be done before we instantiate any RL agents.
ray.init(num_cpus=3, ignore_reinit_error=True, log_to_driver=False)

2026-04-07 03:01:26,202	INFO worker.py:2013 -- Started a local Ray instance.
/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Python version:,3.12.13
Ray version:,2.54.1


Instantiate a PPOTrainer object. We pass in a config object that specifies how the network and training procedure should be configured. Some of the parameters are the following.

- `num_workers` is the number of actors that the agent will create. This determines the degree of parallelism that will be used.
- `num_sgd_iter` is the number of epochs of SGD (passes through the data) that will be used to optimize the PPO surrogate objective at each iteration of PPO.
- `sgd_minibatch_size` is the SGD batch size that will be used to optimize the PPO surrogate objective.
- `model` contains a dictionary of parameters describing the neural net used to parameterize the policy. The `fcnet_hiddens` parameter is a list of the sizes of the hidden layers.

In [4]:
config = PPOConfig() # Create the base config object
config.num_workers = 1
config.num_sgd_iter = 30
config.sgd_minibatch_size = 128
config.num_cpus_per_env_runner = 0  # This avoids running out of resources in the notebook environment when this cell is re-executed

# Configure the model using the new API
config = config.rl_module(model_config={"fcnet_hiddens": [100, 100]})

# Build the trainer using the new API
agent = config.environment('CartPole-v0').build_algo()

2026-04-07 03:01:27,352	WARNING algorithm_config.py:5131 -- You are running PPO on the new API stack! This is the new default behavior for this algorithm. If you don't want to use the new API stack, set `config.api_stack(enable_rl_module_and_learner=False,enable_env_runner_and_connector_v2=False)`. For a detailed migration guide, see here: https://docs.ray.io/en/master/rllib/new-api-stack-migration-guide.html
2026-04-07 03:01:33,993	WARNING rl_module.py:463 -- DeprecationWarning: `RLModule(config=[RLModuleConfig object])` has been deprecated. Use `RLModule(observation_space=.., action_space=.., inference_only=.., model_config=.., catalog_class=..)` instead. This will raise an error in the future!
2026-04-07 03:01:37,745	INFO trainable.py:161 -- Trainable.setup took 10.383 seconds. If your trainable is slow to initialize, consider setting reuse_actors=True to reduce actor creation overheads.
2026-04-07 03:01:37,746	WARNING util.py:62 -- Install gputil for GPU system monitoring.


In [5]:
config = PPOConfig()
config.num_workers = 1
config.num_sgd_iter = 30
config.sgd_minibatch_size = 128
config.num_cpus_per_env_runner = 0  # This avoids running out of resources in the notebook environment when this cell is re-executed
config.rl_module(model_config={"fcnet_hiddens": [100, 100]})
config.resources(num_gpus=1)
config.environment('CartPole-v1')
algo = config.build()
# algo.train()

2026-04-07 03:01:38,184	WARNING 3663525839.py:9 -- DeprecationWarning: `build` has been deprecated. Use `AlgorithmConfig.build_algo` instead. This will raise an error in the future!
2026-04-07 03:01:45,825	WARNING util.py:62 -- Install gputil for GPU system monitoring.


Train the policy on the `CartPole-v1` environment for 2 steps. The CartPole problem is described at https://gymnasium.farama.org/environments/classic_control/cart_pole/.

**EXERCISE:** Inspect how well the policy is doing by looking for the lines that say something like

```
episode_len_mean: 22.262569832402235
episode_reward_mean: 22.262569832402235
```

This indicates how much reward the policy is receiving and how many time steps of the environment the policy ran. The maximum possible reward for this problem is 200. The reward and trajectory length are very close because the agent receives a reward of one for every time step that it survives (however, that is specific to this environment).

In [6]:
for i in range(2):
    result = algo.train()
    print(pretty_print(result))

date: 2026-04-07_03-02-15
done: false
env_runner_group:
  actor_manager_num_outstanding_async_reqs: 2
env_runners:
  agent_episode_return_mean:
    default_agent: 19.84
  env_reset_timer: 0.0002682900000081645
  env_step_timer: 5.465646656046176e-05
  env_to_module_connector:
    connector_pipeline_timer: 0.0001591114486551349
    timers:
      connectors:
        add_observations_from_episodes_to_batch: 7.581480802151123e-06
        add_states_from_episodes_to_batch: 1.557843466315682e-06
        add_time_dim_to_batch_and_zero_pad: 2.470879178476658e-06
        batch_individual_items: 1.5646359714294585e-05
        numpy_to_tensor: 3.297979287614505e-05
  env_to_module_sum_episodes_length_in: 10.588338934082262
  env_to_module_sum_episodes_length_out: 10.588338934082262
  episode_duration_sec_mean: 0.017061258569998756
  episode_len_max: 80
  episode_len_mean: 19.84
  episode_len_min: 8
  episode_return_max: 80.0
  episode_return_mean: 19.84
  episode_return_min: 8.0
  module_episode_

**EXERCISE:** The current network and training configuration are too large and heavy-duty for a simple problem like CartPole. Modify the configuration to use a smaller network and to speed up the optimization of the surrogate objective (fewer SGD iterations and a larger batch size should help).

In [7]:
config = PPOConfig()
config.num_workers = 3
config.num_sgd_iter = 30
config.sgd_minibatch_size = 128
# config['model']['fcnet_hiddens'] = [100, 100]
config.rl_module(model_config={"fcnet_hiddens": [100, 100]})
config.num_cpus_per_env_runner = 0
config.resources(num_gpus=1)

agent = config.environment('CartPole-v1').build()

2026-04-07 03:02:38,050	WARNING util.py:62 -- Install gputil for GPU system monitoring.


**EXERCISE:** Train the agent and try to get a reward of 200. If it's training too slowly you may need to modify the config above to use fewer hidden units, a larger `sgd_minibatch_size`, a smaller `num_sgd_iter`, or a larger `num_workers`.

This should take around 20 or 30 training iterations.

In [8]:
for i in range(20):
    print(f"Iteration {i}")
    result = agent.train()
print(pretty_print(result))

Iteration 0
Iteration 1
Iteration 2
Iteration 3
Iteration 4
Iteration 5
Iteration 6
Iteration 7
Iteration 8
Iteration 9
Iteration 10
Iteration 11
Iteration 12
Iteration 13
Iteration 14
Iteration 15
Iteration 16
Iteration 17
Iteration 18
Iteration 19
date: 2026-04-07_03-04-28
done: false
env_runner_group:
  actor_manager_num_outstanding_async_reqs: 2
env_runners:
  agent_episode_return_mean:
    default_agent: 500.0
  env_reset_timer: .nan
  env_step_timer: 5.231497656020711e-05
  env_to_module_connector:
    connector_pipeline_timer: 0.00015399081331118547
    timers:
      connectors:
        add_observations_from_episodes_to_batch: 7.365839086314517e-06
        add_states_from_episodes_to_batch: 1.6264998565248001e-06
        add_time_dim_to_batch_and_zero_pad: 2.485270072532004e-06
        batch_individual_items: 1.516605098036991e-05
        numpy_to_tensor: 3.090423731374146e-05
  env_to_module_sum_episodes_length_in: 162.23366957308426
  env_to_module_sum_episodes_length_out: 162

Checkpoint the current model. The call to `agent.save()` returns the path to the checkpointed model and can be used later to restore the model.

In [9]:
checkpoint_path = agent.save()
print(checkpoint_path)

TrainingResult(checkpoint=Checkpoint(filesystem=local, path=/tmp/tmpbwwgon_j), metrics={'timers': {'training_iteration': 5.216697775, 'restore_env_runners': 7.615999948029639e-06, 'training_step': 5.216483068999992, 'env_runner_sampling_timer': 1.674145995999993, 'learner_update_timer': 3.5400464030000194, 'synch_weights': 0.001774722999982714, 'synch_env_connectors': 0.0015296589999707066}, 'env_runners': {'env_to_module_connector': {'timers': {'connectors': {'add_time_dim_to_batch_and_zero_pad': np.float64(2.485270072532004e-06), 'add_states_from_episodes_to_batch': np.float64(1.6264998565248001e-06), 'batch_individual_items': np.float64(1.516605098036991e-05), 'add_observations_from_episodes_to_batch': np.float64(7.365839086314517e-06), 'numpy_to_tensor': np.float64(3.090423731374146e-05)}}, 'connector_pipeline_timer': np.float64(0.00015399081331118547)}, 'episode_len_max': np.int64(500), 'module_to_env_connector': {'timers': {'connectors': {'un_batch_to_individual_items': np.float6

Now let's use the trained policy to make predictions.

**NOTE:** Here we are loading the trained policy in the same process, but in practice, this would often be done in a different process (probably on a different machine).

In [11]:
trained_config = config.copy()

test_agent = trained_config.environment('CartPole-v1').build()
test_agent.restore(checkpoint_path)

2026-04-07 03:05:31,076	WARNING util.py:62 -- Install gputil for GPU system monitoring.
2026-04-07 03:05:31,085	INFO trainable.py:578 -- Restored on 172.28.0.12 from checkpoint: Checkpoint(filesystem=local, path=/tmp/tmpbwwgon_j)


Now use the trained policy to act in an environment. The key line is the call to `test_agent.compute_action(state)` which uses the trained policy to choose an action.

**EXERCISE:** Verify that the reward received roughly matches up with the reward printed in the training logs.

In [14]:
import torch
env = gym.make('CartPole-v1', disable_env_checker=True)
state, _ = env.reset() # Unpack the tuple, get only the observation array
done = False
cumulative_reward = 0
import numpy as np

# Get the RLModule instance for the default policy
# The module ID is typically 'default_policy' for single-agent setups
rl_module = test_agent.get_module(module_id="default_policy")

while not done:
    # Prepare the observation as a batch for forward_inference
    # forward_inference expects a dictionary with an 'obs' key
    obs_batch = {'obs': torch.from_numpy(np.array([state])).float()} # 'state' is now correctly the observation array

    # Compute the action using the RLModule's forward_inference method
    action_output = rl_module.forward_inference(obs_batch)

    # Extract the scalar action from the output tensor by taking argmax of action_dist_inputs
    # CartPole has discrete actions (0 or 1), so action_dist_inputs are logits
    action = torch.argmax(action_output['action_dist_inputs'], dim=1).item()

    # env.step returns (observation, reward, terminated, truncated, info)
    state, reward, terminated, truncated, _ = env.step(action) # 'state' is updated with the new observation array
    done = terminated or truncated
    cumulative_reward += reward

print(cumulative_reward)

500.0


# RLlib Exercise 2 - Custom Environments and Reward Shaping

**GOAL:** The goal of this exercise is to demonstrate how to adapt your own problem to use RLlib.

To understand how to use **RLlib**, see the documentation at http://rllib.io.

RLlib is not only easy to use in simulated benchmarks but also in the real-world. Here, we will cover two important concepts: how to create your own Markov Decision Process abstraction, and how to shape the reward of your environment so make your agent more effective.

In [3]:
! pip install -U ray[rllib]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 54.6 MB/s eta 0:00:00
  Attempting uninstall: gymnasium
    Found existing installation: gymnasium 1.2.3
    Uninstalling gymnasium-1.2.3:
      Successfully uninstalled gymnasium-1.2.3


In [1]:
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import gymnasium
from gymnasium import spaces
import numpy as np
from tutorial.rllib_exercises import test_exercises

import ray
from ray.rllib.algorithms.ppo import PPO, PPOConfig
ray.init(ignore_reinit_error=True, log_to_driver=False)

2026-04-07 03:09:55,808	INFO worker.py:2013 -- Started a local Ray instance.
/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Python version:,3.12.13
Ray version:,2.54.1


## 1. Different Spaces

The first thing to do when formulating an RL problem is to specify the dimensions of your observation space and action space. Abstractions for these are provided in ``gym``.

### **Exercise 1:** Match different actions to their corresponding space.

The purpose of this exercise is to familiarize you with different Gym spaces. For example:

    discrete = spaces.Discrete(10)
    print("Random sample of this space: ", [discrete.sample() for i in range(4)])

Use `help(spaces)` or `help([specific space])` (i.e., `help(spaces.Discrete)`) for more info.

In [4]:
action_space_map = {
    "discrete_10": spaces.Discrete(10),
    "box_1": spaces.Box(0, 1, shape=(1,)),
    "box_3x1": spaces.Box(-2, 2, shape=(3, 1)),
    "multi_discrete": spaces.MultiDiscrete([ 5, 2, 2, 4 ])
}

action_space_jumble = {
    "discrete_10": 1,
    "multi_discrete": np.array([0, 0, 0, 2]),
    "box_3x1": np.array([[-1.2657754], [-1.6528835], [ 0.5982418]], dtype=np.float32),
    "box_1": np.array([0.89089584], dtype=np.float32),
}
# help(spaces.Box)

for space_id, state in action_space_jumble.items():
    print(action_space_map[space_id], state)
    assert action_space_map[space_id].contains(state), (
        "Looks like {} to {} is matched incorrectly.".format(space_id, state))

print("Success!")

Discrete(10) 1
MultiDiscrete([5 2 2 4]) [0 0 0 2]
Box(-2.0, 2.0, (3, 1), float32) [[-1.2657754]
 [-1.6528835]
 [ 0.5982418]]
Box(0.0, 1.0, (1,), float32) [0.89089584]
Success!


## **Exercise 2**: Setting up a custom environment with rewards

We'll setup an `n-Chain` environment, which presents moves along a linear chain of states, with two actions:

     (0) forward, which moves along the chain but returns no reward
     (1) backward, which returns to the beginning and has a small reward

The end of the chain, however, presents a large reward, and by moving 'forward', at the end of the chain this large reward can be repeated.

#### Step 1: Implement ``ChainEnv._setup_spaces``

We'll use a `spaces.Discrete` action space and observation space. Implement `ChainEnv._setup_spaces` so that `self.action_space` and `self.obseration_space` are proper gym spaces.
  
1. Observation space is an integer in ``[0 to n-1]``.
2. Action space is an integer in ``[0, 1]``.

For example:

```python
    self.action_space = spaces.Discrete(2)
    self.observation_space = ...
```

You should see a message indicating tests passing when done correctly!

#### Step 2: Implement a reward function.

When `env.step` is called, it returns a tuple of ``(state, reward, done, info)``. Right now, the reward is always 0.

Implement it so that

1. ``action == 1`` will return `self.small_reward`.
2. ``action == 0`` will return 0 if `self.state < self.n - 1`.
3. ``action == 0`` will return `self.large_reward` if `self.state == self.n - 1`.

You should see a message indicating tests passing when done correctly.

In [10]:
class ChainEnv(gymnasium.Env):

    def __init__(self, env_config = None):
        env_config = env_config or {}
        self.n = env_config.get("n", 20)
        self.small_reward = env_config.get("small", 2)  # payout for 'backwards' action
        self.large_reward = env_config.get("large", 10)  # payout at end of chain for 'forwards' action
        self.state = 0  # Start at beginning of the chain
        self._horizon = self.n
        self._counter = 0  # For terminating the episode
        self._setup_spaces()

    def _setup_spaces(self):
        ##############
        # TODO: Implement this so that it passes tests
        self.action_space = spaces.Discrete(2)
        self.observation_space = spaces.Discrete(self.n)
        ##############

    def step(self, action):
        assert self.action_space.contains(action)
        if action == 1:  # 'backwards': go back to the beginning, get small reward
            ##############
            # TODO 2: Implement this so that it passes tests
            reward = self.small_reward
            ##############
            self.state = 0
        elif self.state < self.n - 1:  # 'forwards': go up along the chain
            ##############
            # TODO 2: Implement this so that it passes tests
            reward = 0
            self.state += 1
        else:  # 'forwards': stay at the end of the chain, collect large reward
            ##############
            # TODO 2: Implement this so that it passes tests
            reward = self.large_reward
            ##############
        self._counter += 1
        terminated = self._counter >= self._horizon
        truncated = False # Add truncated as it's part of the new step API
        return self.state, reward, terminated, truncated, {}

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = 0
        self._counter = 0
        return self.state, {} # reset returns (observation, info) in gymnasium

# Tests here:
test_exercises.test_chain_env_spaces(ChainEnv)
test_exercises.test_chain_env_reward(ChainEnv)

Testing if spaces have been setup correctly...
Success! You've setup the spaces correctly.
Testing if reward has been setup correctly...
Success! You've setup the rewards correctly.


### Now let's train a policy on the environment and evaluate this policy on our environment.

You'll see that despite an extremely high reward, the policy has barely explored the state space.

In [23]:
from ray.rllib.connectors.env_to_module import FlattenObservations
trainer_config = PPOConfig().env_runners(
        # This transforms Discrete(20) into a one-hot vector of length 20
        env_to_module_connector=lambda env: FlattenObservations(),
    ).api_stack(
        enable_rl_module_and_learner=True,
        enable_env_runner_and_connector_v2=True,
    )
trainer_config.num_workers = 1
trainer_config.train_batch_size = 400
trainer_config.sgd_minibatch_size = 64
trainer_config.num_sgd_iter = 10
# Avoid running out of resources in the notebook environment when re-executing cells.
trainer_config.num_cpus_per_env_runner = 0
# trainer_config.resources(num_gpus=1)

# For Discrete observation space, we need an explicit encoder configuration.
# This should be nested within the 'model_config' dictionary.
trainer_config = trainer_config.rl_module(
    model_config={
        "fcnet_hiddens": [64, 64], # These are the hidden layers *after* the encoder
        # "observation_encoder_config": {
        #     "output_dim": 64, # Output dimension of the observation encoder (embedding size)
        #     "encoder_type": "embedding" # Explicitly specify that an embedding layer should be used
        # }
    }
)



agent = trainer_config.environment(ChainEnv).build_algo()

(raylet) WARNING: 12 PYTHON worker processes have been started on node: ac202ccc1f717a4567ccc543efd4f085767371c83a2089b59078026e with address: 172.28.0.12. This could be a result of using a large number of actors, or due to tasks blocked in ray.get() calls (see https://github.com/ray-project/ray/issues/3644 for some discussion of workarounds).


2026-04-07 03:38:52,639	INFO trainable.py:161 -- Trainable.setup took 24.339 seconds. If your trainable is slow to initialize, consider setting reuse_actors=True to reduce actor creation overheads.
2026-04-07 03:38:52,643	WARNING util.py:62 -- Install gputil for GPU system monitoring.


In [15]:
from ray.tune.logger import pretty_print
for i in range(2):
    print(f"Iteration {i}")
    result = agent.train()
print(pretty_print(result))

Iteration 0
Iteration 1
date: 2026-04-07_03-22-37
done: false
env_runner_group:
  actor_manager_num_outstanding_async_reqs: 2
env_runners:
  agent_episode_return_mean:
    default_agent: 28.8
  env_reset_timer: .nan
  env_step_timer: 9.59421460420561e-05
  env_to_module_connector:
    connector_pipeline_timer: 0.0007769286138552554
    timers:
      connectors:
        add_observations_from_episodes_to_batch: 1.8687179931480345e-05
        add_states_from_episodes_to_batch: 5.662846922086958e-06
        add_time_dim_to_batch_and_zero_pad: 9.05127076287591e-06
        batch_individual_items: 5.377440736783948e-05
        flatten_observations: 0.00011407811288736126
        numpy_to_tensor: 8.206163320761132e-05
  env_to_module_sum_episodes_length_in: 9.111931220596055
  env_to_module_sum_episodes_length_out: 9.111931220596055
  episode_duration_sec_mean: 0.058487549700043925
  episode_len_max: 20
  episode_len_mean: 20.0
  episode_len_min: 20
  episode_return_max: 38.0
  episode_return_

## Exercise 3: Shaping the reward to encourage proper behavior.

You'll see that despite an extremely high reward, the policy has barely explored the state space. This is often the situation - where the reward designed to encourage a particular solution is suboptimal, and the behavior created is unintended.

#### Modify `ShapedChainEnv.step` to provide a reward that encourages the policy to traverse the chain (not just stick to 0). Do not change the behavior of the environment (the action -> state behavior should be the same).

You can change the reward to be whatever you wish.

In [16]:
class ShapedChainEnv(ChainEnv):
    def step(self, action):
        assert self.action_space.contains(action)
        if action == 1:  # 'backwards': go back to the beginning
            reward = -1
            self.state = 0
        elif self.state < self.n - 1:  # 'forwards': go up along the chain
            reward = 1
            self.state += 1
        else:  # 'forwards': stay at the end of the chain
            reward = -1
        self._counter += 1
        terminated = self._counter >= self._horizon
        truncated = False
        return self.state, reward, terminated, truncated, {}

### Evaluate `ShapedChainEnv` by running the cell below.

This trains PPO on the new env and counts the number of states seen.

In [25]:
import torch
import numpy as np

trainer = trainer_config.environment(ShapedChainEnv).build_algo()
for i in range(20):
    print("Training iteration {}...".format(i))
    trainer.train()

env = ShapedChainEnv({})

max_states = []
# Get the RLModule instance for the default policy once outside the loop
rl_module = trainer.get_module(module_id="default_policy")

for i in range(5):
    state, _ = env.reset() # Correctly unpack (observation, info)
    terminated = False
    truncated = False
    done = False # Initialize done flag
    max_state = -1
    cumulative_reward = 0
    while not done:
        # Convert integer state to one-hot vector for the RLModule
        one_hot_state = np.zeros(env.n, dtype=np.float32)
        one_hot_state[state] = 1.0

        # Prepare the observation as a batch for forward_inference, add batch dim
        obs_batch = {'obs': torch.from_numpy(one_hot_state).float().unsqueeze(0)}

        # Compute the action using the RLModule's forward_inference method
        action_output = rl_module.forward_inference(obs_batch)

        # Extract the scalar action by taking argmax of action_dist_inputs
        action = torch.argmax(action_output['action_dist_inputs'], dim=1).item()

        # env.step returns (observation, reward, terminated, truncated, info)
        state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        max_state = max(max_state, state)
        cumulative_reward += reward
    max_states += [max_state]

print("Cumulative reward you've received is: {}!".format(cumulative_reward))
print("Max state you've visited is: {}. This is out of {} states.".format(np.mean(max_state), env.n))
assert (env.n - np.mean(max_state)) / env.n < 0.2, "This policy did not traverse many states."


2026-04-07 03:41:06,576	INFO trainable.py:161 -- Trainable.setup took 25.937 seconds. If your trainable is slow to initialize, consider setting reuse_actors=True to reduce actor creation overheads.
2026-04-07 03:41:06,578	WARNING util.py:62 -- Install gputil for GPU system monitoring.


Training iteration 0...
Training iteration 1...
Training iteration 2...
Training iteration 3...
Training iteration 4...
Training iteration 5...
Training iteration 6...
Training iteration 7...
Training iteration 8...
Training iteration 9...
Training iteration 10...
Training iteration 11...
Training iteration 12...
Training iteration 13...
Training iteration 14...
Training iteration 15...
Training iteration 16...
Training iteration 17...
Training iteration 18...
Training iteration 19...
Cumulative reward you've received is: 18!
Max state you've visited is: 19.0. This is out of 20 states.
